# Анализ структуры и динамики международной мобильности россиийских студентов (SQL)

**Проект:** Анализ международной мобильности российских студентов (2000-2024)

## Задачи
Дполнить анализ, проведенный в Python:
- проанализировать структуру данных; 
- рассчитать показатели структуры и динамики международной мобильности студентов из России с 2000 по 2024 г. (на основе данных по 110 странам).

## Данные
- очищенный в Python исходный датасет, содержащий данные по 110 странам (student_mobility.csv);
- дополнительно: очищенный в Python исходный датасет, в который для каждой страны добавлен соответствующий регион мира (student_mobility_regions.csv).

Для удобства анализа в таблицах изменены названия колонок. Анализ выполнен в PostgreSQL, все результаты запросов сохранены в формате csv. 

Ниже воспроизведен весь анализ с комментариями.

In [1]:
import pandas as pd

# Основной датасет
df=pd.read_csv('../data/processed/student_mobility_clean.csv')
df_sql=df.rename(columns={'Страна':'country',
                       'Год':'year',
                       'Число студентов из РФ':'students'})
df_sql.to_csv('../sql/student_mobility.csv', index=False)

# Дополнительно: датасет с регионами мира
df_regions=pd.read_csv('../data/processed/student_mobility_clean_regions.csv')
df_regions=df_regions.rename(columns={'Страна':'country',
                       'Год':'year',
                       'Число студентов из РФ':'students',
                        'Регион': 'region'})
df_regions.to_csv('../sql/student_mobility_regions.csv', index=False)

## Структура и полнота данных

- **Структура данных**

```
SELECT 
    COUNT(*) AS Всего_строк,
    COUNT(DISTINCT country) AS Всего_стран,
    COUNT(DISTINCT year) AS Всего_лет,
    MIN(year) AS Первый_год,
    MAX(year) AS Последний_год
FROM student_mobility;
```
Результат запроса: <span style="color:red">01_data_structure.csv</span>

In [2]:
data_structure=pd.read_csv('01_data_structure.csv')
data_structure

,Всего_строк,Всего_стран,Всего_лет,Первый_год,Последний_год
0,1805,110,25,2000,2024


- **Полнота данных**
```
SELECT 
    country AS Страна,
    MIN(year) AS Первый_год,
    MAX(year) AS Последний_год,
    COUNT(*) AS Число_наблюдений,
    25 AS Ожидаемое_число_наблюдений,
    (25 - COUNT(*)) AS Число_пропусков,
    ROUND((25 - COUNT(*)) * 100.0 / 25) || '%' AS Доля_пропусков,
    CASE 
        WHEN COUNT(*) = 25 THEN 'Полные данные'
        WHEN COUNT(*) = (MAX(year) - MIN(year) + 1) AND COUNT(*) >= 5 
        THEN 'Непрерывный ряд (5+ лет)'
        WHEN COUNT(*) = (MAX(year) - MIN(year) + 1) AND COUNT(*) BETWEEN 2 AND 4 
        THEN 'Непрерывный ряд (2-4 года)'
        WHEN COUNT(*) = (MAX(year) - MIN(year) + 1) AND COUNT(*) = 1 
        THEN 'Данные за 1 год'
        ELSE 'Пропуски внутри периода'
    END AS Полнота_данных
FROM student_mobility
GROUP BY country
ORDER BY ROUND((25 - COUNT(*)) * 100.0 / 25) DESC;
```

Результат запроса: <span style="color:red">02_data_completeness.csv</span>

In [3]:
data_completeness=pd.read_csv('02_data_completeness.csv')
data_completeness.head(10)

,Страна,Первый_год,Последний_год,Число_наблюдений,Ожидаемое_число_наблюдений,Число_пропусков,Доля_пропусков,Полнота_данных
0,"Korea, Democratic People's Republic of",2009,2009,1,25,24,96%,Данные за 1 год
1,Nicaragua,2023,2023,1,25,24,96%,Данные за 1 год
2,Guyana,2010,2010,1,25,24,96%,Данные за 1 год
3,Kuwait,2008,2009,2,25,23,92%,Непрерывный ряд (2-4 года)
4,Senegal,2023,2024,2,25,23,92%,Непрерывный ряд (2-4 года)
5,Guatemala,2019,2023,2,25,23,92%,Пропуски внутри периода
6,United Arab Emirates,2011,2012,2,25,23,92%,Непрерывный ряд (2-4 года)
7,Congo,2003,2013,3,25,22,88%,Пропуски внутри периода
8,Turkmenistan,2014,2021,4,25,21,84%,Пропуски внутри периода
9,Costa Rica,2004,2018,4,25,21,84%,Пропуски внутри периода


Данные отсортированы по убыванию доли пропущенных значений.

Выделены категории данных по полноте:  

*Полные данные*: есть данные по числу российских студентов в стране за все годы с 2000 по 2024.  
*Пропуски внутри периода*: данные за отдельные годы внутри временного ряда отсутствуют.  
*Непрерывный ряд (5+ лет)*: данные за неполный период, временной ряд состоит из 5 и более лет без пропусков. 
*Непрерывный ряд (2-4 года)*: данные за неполный период, временной ряд состоит из 2-4 лет без пропусков.  
*Данные за 1 год*: единичное наблюдение.

Датасет содержит информацию по 110 странам за 25 лет с 2000 по 2024 г. Полные данные есть только по 11 из 110 стран, 10 из них европейские. 1/3 стран имеет долю пропусков 50% и выше. В их числе Германия с долей пропусков 52%. Доля пропусков в данных по США еще выше и равна 60%.

## Основные статистические показатели

- **Всего студентов, среднее, минимальное и максимальное число, размах, стандартное отклонение и коэффициент вариации**
```
SELECT country AS  Страна,
    MIN(year) AS Первый_год,
    MAX(year) AS Последний_год,
    COUNT(*) AS Число_лет,
    SUM(students) AS Всего_студентов,
    ROUND(AVG(students)) AS Среднее_число,
    MIN(students) AS Мин_число,
    MAX(students) AS Макс_число,
    MAX(students)-MIN(students) AS Размах,
    ROUND(STDDEV(students),1) AS Ст_отклонение,
    ROUND(STDDEV(students)/AVG(students),2) AS Коэффициент_вариации
FROM student_mobility
GROUP BY country
ORDER BY Всего_студентов DESC;
```

Результат запроса сохранен в файл: <span style="color:red">03_stats.csv</span>

In [4]:
stats=pd.read_csv('03_stats.csv')
stats.head(10)

,Страна,Первый_год,Последний_год,Число_лет,Всего_студентов,Среднее_число,Мин_число,Макс_число,Размах,Ст_отклонение,Коэффициент_вариации
0,Germany,2013,2024,12,123770,10314,9480,11369,1889,730.6,0.07
1,Czechia,2000,2023,24,79907,3329,110,8255,8145,2742.1,0.82
2,France,2000,2024,25,76612,3064,1453,4300,2847,716.6,0.23
3,United Kingdom,2000,2023,24,69804,2909,1058,4092,3034,926.6,0.32
4,United States,2005,2023,10,49296,4930,4466,5299,833,285.2,0.06
5,Ukraine,2005,2024,19,47199,2484,623,4734,4111,1566.7,0.63
6,Finland,2000,2024,24,42314,1763,656,2799,2143,648.4,0.37
7,Kazakhstan,2000,2024,21,39012,1858,728,3600,2872,811.3,0.44
8,Belarus,2002,2024,23,35264,1533,0,2622,2622,980.2,0.64
9,Italy,2000,2023,24,33592,1400,118,3047,2929,845.3,0.60


Результат отсортирован по общему числу студентов по странам за весь период.

Первые пять лидеров по входящей образовательной мобильности из России: Германия, Чехия, Франция, Великобритания и США. Абсолютный лидер - Германия с общим числом студентов 123770 человек. Если рассматривать среднее число студентов в год, то США поднимаются на второе место после Германии. 

Среди лидеров по входящей мобильности наиболее стабильные образовательные потоки характерны для США (CV=0,06) и Германии (CV=0,07), в то время как поток в Чехию наиболее изменчив (CV=0,82).

## Динамика мобильности

Для удобства результаты первых трех запросов этой части анализа отсортированы сначала по 10 ведущим странам Европы и Азии по числу студентов из России (эти страны подробно рассматриваются в основной части проекта), а после этого - по остальным странам в алфавитном порядке. 

- **Годовое изменение числа студентов по странам**  

```
SELECT
    country AS Страна,
    year AS Год,
    students AS Число_студентов,
    students - LAG(students) OVER (
        PARTITION BY country ORDER BY year
    ) AS Годовое_изменение
FROM student_mobility
ORDER BY
    CASE country
        WHEN 'Germany' THEN 1
        WHEN 'Czechia' THEN 2
        WHEN 'France' THEN 3
        WHEN 'United Kingdom' THEN 4
        WHEN 'Finland' THEN 5
        WHEN 'Kazakhstan' THEN 6
        WHEN 'Armenia' THEN 7
        WHEN 'Kyrgyzstan' THEN 8
        WHEN 'Türkiye' THEN 9
        WHEN 'Korea, Republic of' THEN 10
    END,
    country,
    year;
```

Результат запроса сохранен в файл: <span style="color:red">04_students_yearly_change.csv</span>

In [5]:
students_yearly_change=pd.read_csv('04_students_yearly_change.csv')
students_yearly_change.head(10)

,Страна,Год,Число_студентов,Годовое_изменение
0,Germany,2013,9480,NaN
1,Germany,2014,9668,188.0
2,Germany,2015,9953,285.0
3,Germany,2016,9711,-242.0
4,Germany,2017,9620,-91.0
5,Germany,2018,10121,501.0
6,Germany,2019,9646,-475.0
7,Germany,2020,11055,1409.0
8,Germany,2021,11149,94.0
9,Germany,2022,11369,220.0


- **Темп роста числа студентов по странам**
```
SELECT 
    country AS Страна,
    year AS Год,
    students AS Число_студентов,
    CASE 
        WHEN LAG(students) OVER (PARTITION BY country ORDER BY year) IS NULL 
            THEN '—'
        WHEN LAG(students) OVER (PARTITION BY country ORDER BY year) = 0 
            AND students > 0
        THEN 'рост с 0 до ' || students || ' студентов'
        WHEN LAG(students) OVER (PARTITION BY country ORDER BY year) = 0 
            AND students = 0
        THEN '0%'
        ELSE ROUND(
            (students - LAG(students) OVER (PARTITION BY country ORDER BY year)) 
            * 100.0 / LAG(students) OVER (PARTITION BY country ORDER BY year)
        , 1) || '%'
    END AS Темп_роста
FROM student_mobility
ORDER BY
    CASE country
        WHEN 'Germany' THEN 1
        WHEN 'Czechia' THEN 2
        WHEN 'France' THEN 3
        WHEN 'United Kingdom' THEN 4
        WHEN 'Finland' THEN 5
        WHEN 'Kazakhstan' THEN 6
        WHEN 'Armenia' THEN 7
        WHEN 'Kyrgyzstan' THEN 8
        WHEN 'Türkiye' THEN 9
        WHEN 'Korea, Republic of' THEN 10
    END,
    country,
    year;
```

Результат запроса сохранен в файл: <span style="color:red">05_students_growth_rate.csv</span>

In [6]:
students_growth_rate=pd.read_csv('05_students_growth_rate.csv')
students_growth_rate.head(10)

,Страна,Год,Число_студентов,Темп_роста
0,Germany,2013,9480,—
1,Germany,2014,9668,2.0%
2,Germany,2015,9953,2.9%
3,Germany,2016,9711,-2.4%
4,Germany,2017,9620,-0.9%
5,Germany,2018,10121,5.2%
6,Germany,2019,9646,-4.7%
7,Germany,2020,11055,14.6%
8,Germany,2021,11149,0.9%
9,Germany,2022,11369,2.0%


- **Совокупный среднегодовой темп роста CAGR по странам за 2000-2024 гг. (доступные данные)**
```
WITH years AS (
    SELECT 
        country,
        MIN(year) AS first_year,
        MAX(year) AS last_year
    FROM student_mobility
    GROUP BY country
),
stud AS(
    SELECT country,
        first_year,
        last_year,
        (SELECT students FROM student_mobility sm
        WHERE sm.country=y.country AND sm.year=y.first_year) AS first_count,
        (SELECT students FROM student_mobility sm
        WHERE sm.country=y.country AND sm.year=y.last_year) AS last_count
    FROM years y
)
SELECT 
    country AS Страна,
    first_year AS Первый_год,
    last_year AS Последний_год,
    CASE 
        WHEN first_count IS NULL OR first_count = 0 OR last_year = first_year
        THEN '—'
        ELSE ROUND((POWER(last_count*1.0/first_count, 1.0/(last_year - first_year)) 
        - 1) * 100, 1) || '%' 
    END AS Среднегодовой_темп_роста
FROM stud
ORDER BY
    CASE country
        WHEN 'Germany' THEN 1
        WHEN 'Czechia' THEN 2
        WHEN 'France' THEN 3
        WHEN 'United Kingdom' THEN 4
        WHEN 'Finland' THEN 5
        WHEN 'Kazakhstan' THEN 6
        WHEN 'Armenia' THEN 7
        WHEN 'Kyrgyzstan' THEN 8
        WHEN 'Türkiye' THEN 9
        WHEN 'Korea, Republic of' THEN 10
    END,
    country;
```

Результат запроса сохранен в файл: <span style="color:red">06_students_cagr.csv</span>

In [7]:
cagr=pd.read_csv('06_cagr.csv')
cagr.head(10)

,Страна,Первый_год,Последний_год,Среднегодовой_темп_роста
0,Germany,2013,2024,1.3%
1,Czechia,2000,2023,20.3%
2,France,2000,2024,3.3%
3,United Kingdom,2000,2023,4.7%
4,Finland,2000,2024,4.9%
5,Kazakhstan,2000,2024,4.6%
6,Armenia,2003,2024,11.3%
7,Kyrgyzstan,2001,2024,5.2%
8,Türkiye,2000,2023,5.0%
9,"Korea, Republic of",2000,2024,12.7%


- **Среднегодовой темп роста по странам за 2013-2024 гг. (гибкий период)**

CAGR рассчитан на основе ближайших доступных значений в диапазонах 2013-2015 и 2022-2024 гг., что позволяет учесть страны, по которым нет данных за 2013 или 2024 г.
```
WITH years_b AS (
    SELECT country,
        region,
        year,
        students,
        ROW_NUMBER() OVER(PARTITION BY country, region ORDER BY year) 
        AS row_num1
    FROM student_mobility_regions
    WHERE year BETWEEN 2013 AND 2015
),
years_e AS (
    SELECT country,
        region,
        year,
        students,
        ROW_NUMBER() OVER(PARTITION BY country, region ORDER BY year DESC) 
        AS row_num2
    FROM student_mobility_regions
    WHERE year BETWEEN 2022 AND 2024
),
first_v AS (
    SELECT country,
        region,
        year AS f_year,
        SUM(students) AS f_value
    FROM years_b
    WHERE row_num1=1
    GROUP BY country, region, year
),
last_v AS (
    SELECT country,
        region,
        year AS l_year,
        SUM(students) AS l_value
    FROM years_e
    WHERE row_num2=1
    GROUP BY country, region, year
)
SELECT f.country AS Страна,
    f.region AS Регион,
    f_year AS Первый_год,
    l_year AS Последний_год,
    f_value AS Первое_значение,
    l_value AS Последнее_значение,
    l_year-f_year AS Число_лет,
    l_value-f_value AS Прирост,
    CASE
        WHEN f_value IS NULL OR l_value IS NULL OR f_value=0 OR l_value=0
        THEN NULL
        ELSE ROUND((POWER(l_value*1.0/f_value, 1.0/(l_year-f_year))-1)*100,1) || '%'
    END AS cagr_2013_2024
FROM first_v f JOIN last_v l ON f.country=l.country AND f.region=l.region
ORDER BY Прирост DESC;
```

Результат запроса сохранен в файл: <span style="color:red">07_cagr_2013_2024.csv</span>

In [8]:
cagr_2013_2024=pd.read_csv('07_cagr_2013_2024.csv')
cagr_2013_2024.head(10)

,Страна,Регион,Первый_год,Последний_год,Первое_значение,Последнее_значение,Число_лет,Прирост,cagr_2013_2024
0,Czechia,Europe,2013,2023,3455,7781,10,4326,8.5%
1,Türkiye,Asia,2013,2023,60,3083,10,3023,48.3%
2,Armenia,Asia,2013,2024,903,2761,11,1858,10.7%
3,Kyrgyzstan,Asia,2013,2024,927,2623,11,1696,9.9%
4,Germany,Europe,2013,2024,9480,10974,11,1494,1.3%
5,Austria,Europe,2013,2024,1004,2345,11,1341,8.0%
6,Slovakia,Europe,2013,2024,63,1253,11,1190,31.2%
7,"Korea, Republic of",Asia,2013,2024,314,1351,11,1037,14.2%
8,Italy,Europe,2013,2023,2103,3047,10,944,3.8%
9,Kazakhstan,Asia,2013,2024,1315,2123,11,808,4.5%


- **Среднегодовые темпы роста Европы и Азии (2000-2024 гг.)**
```
WITH region_sums AS (
    SELECT region,
        SUM(CASE WHEN year = 2000 THEN students END) AS sum_2000,
        SUM(CASE WHEN year = 2024 THEN students END) AS sum_2024
    FROM student_mobility_regions
    WHERE region IN ('Europe', 'Asia')
    GROUP BY region
)
SELECT region AS Регион,
    ROUND((POWER(sum_2024 * 1.0 /sum_2000, 1.0 / (2024-2000)) - 1) * 100, 1) || '%' AS Среднегодовой_темп_роста
FROM region_sums
ORDER BY 1 DESC;
```

Результат запроса сохранен в файл: <span style="color:red">08_europe_asia_cagr.csv</span>

In [9]:
europe_asia_cagr=pd.read_csv('08_europe_asia_cagr.csv')
europe_asia_cagr

,Регион,Среднегодовой_темп_роста
0,Europe,6.4%
1,Asia,7.0%


- **Среднегодовые темпы роста Европы и Азии за два периода (2000-2012 и 2013-2024 гг.)**
```
WITH region_sums_2000_2012 AS (
    SELECT region,
        SUM(CASE WHEN year=2000 THEN students END) AS sum_2000,
        SUM(CASE WHEN year = 2012 THEN students END) AS sum_2012
    FROM student_mobility_regions
    WHERE region IN ('Europe', 'Asia')
    GROUP BY region
),
cagr_2000_2012 AS (
    SELECT region AS Регион,
        ROUND((POWER(sum_2012 * 1.0 /sum_2000, 1.0 / (2012-2000)) - 1) * 100, 1) || '%' AS cagr_2000_2012
    FROM region_sums_2000_2012
),
region_sums_2013_2024 AS (
    SELECT region,
        SUM(CASE WHEN year=2013 THEN students END) AS sum_2013,
        SUM(CASE WHEN year = 2024 THEN students END) AS sum_2024
    FROM student_mobility_regions
    WHERE region IN ('Europe', 'Asia')
    GROUP BY region
),
cagr_2013_2024 AS (
    SELECT region AS Регион,
        ROUND((POWER(sum_2024 * 1.0 /sum_2013, 1.0 / (2024-2013)) - 1) * 100, 1) || '%' AS cagr_2013_2024
    FROM region_sums_2013_2024
)
SELECT Регион,
    cagr_2000_2012,
    cagr_2013_2024
FROM cagr_2000_2012 JOIN cagr_2013_2024 USING (Регион)
ORDER BY 1 DESC;
```

Результат запроса сохранен в файл: <span style="color:red">09_europe_asia_cagr_2_periods.csv</span>

In [10]:
europe_asia_cagr_2_periods=pd.read_csv('09_europe_asia_cagr_2_periods.csv')
europe_asia_cagr_2_periods

,Регион,cagr_2000_2012,cagr_2013_2024
0,Europe,12.9%,-2.1%
1,Asia,9.0%,6.1%


Среднегодовые темпы роста ведущих стран Европы и Азии:  
Чехия, Италия, Армения и Республика Корея показывают наиболее высокие среднегодовые темпы роста за весь период с 2000 по 2024 г., но актуальные лидеры роста в Европе и Азии (2013-2024 гг.) – Турция, Словакия, Сербия, Словения, Венгрия, Узбекистан, Республика Корея.
Среднегодовой темп роста Германии составляет 1,3%.

Среднегодовые темпы роста регионов в 2000-2024 гг. показывают, что мобильность студентов из России быстрее росла в азиатском направлении, чем в европейском: Европа (6,4%), Азия (7%).

Чтобы увидеть, были ли изменения в динамике роста Европы и Азии за 25 лет, дополнительно рассчитаны среднегодовые темпы роста по каждому региону за два более коротких периода: 2000–2012 гг. и 2013–2024 гг. Полученные результаты подтверждают предположение об изменении тренда: в течение 25 лет Европа от быстрого роста переходит к снижению, а Азия продолжает расти, но темп роста снижается. Несмотря опережение Европы по темпу роста, вклад стран Азии в общий образовательный поток российских студентов за границу остается существенно ниже, чем у европейских стран.


## Структура мобильности 

Результаты первых двух запросов отсортированы по убыванию доли стран в общем потоке.

- **Структура в начале рассматриваемого периода (2000 г.)**
```
SELECT country AS Страна,
    students AS Число_студентов_начало_периода,
    ROUND(students*100.0/(
        SELECT SUM(students) 
        FROM student_mobility 
        WHERE year=(
            SELECT MIN(year) FROM student_mobility)
    ),1) || '%' AS Доля_страны
FROM student_mobility
WHERE year=(
    SELECT MIN(year) FROM student_mobility)
ORDER BY ROUND(students*100.0/(
    SELECT SUM(students) 
    FROM student_mobility 
    WHERE year=(
        SELECT MIN(year) FROM student_mobility)
),1) DESC;
```

Результат запроса сохранен в файл: <span style="color:red">10_country_shares_2000.csv</span>

In [11]:
country_shares_2000=pd.read_csv('10_country_shares_2000.csv')
country_shares_2000.head(10)

,Страна,Число_студентов_начало_периода,Доля_страны
0,France,1453,15.6%
1,United Kingdom,1058,11.4%
2,Türkiye,1004,10.8%
3,Kazakhstan,728,7.8%
4,Finland,656,7.1%
5,Sweden,462,5.0%
6,Latvia,430,4.6%
7,Norway,339,3.7%
8,Switzerland,327,3.5%
9,Austria,300,3.2%


- **Структура в конце рассматриваемого периода (2024 г.)**
```
SELECT country AS Страна,
    students AS Число_студентов_конец_периода,
    ROUND(students*100.0/(
        SELECT SUM(students) 
        FROM student_mobility 
        WHERE year=(
            SELECT MAX(year) FROM student_mobility)
    ),1) || '%' AS Доля_страны
FROM student_mobility
WHERE year=(
    SELECT MAX(year) FROM student_mobility)
ORDER BY ROUND(students*100.0/(
    SELECT SUM(students) 
    FROM student_mobility 
    WHERE year=(
        SELECT MAX(year) FROM student_mobility)
),1) DESC;
```

Результат запроса сохранен в файл: <span style="color:red">11_country_shares_2024.csv</span>

In [12]:
country_shares_2024=pd.read_csv('11_country_shares_2024.csv')
country_shares_2024.head(10)

,Страна,Число_студентов_конец_периода,Доля_страны
0,Germany,10974,26.1%
1,France,3159,7.5%
2,Armenia,2761,6.6%
3,Kyrgyzstan,2623,6.2%
4,Austria,2345,5.6%
5,Kazakhstan,2123,5.1%
6,Finland,2066,4.9%
7,Belarus,1901,4.5%
8,"Korea, Republic of",1351,3.2%
9,Slovakia,1253,3.0%


- **Уровень концентрации мобильности по странам (HHI) в 2000 и 2024 гг. и его изменение**
```
WITH total_2000 AS (
    SELECT SUM(students) AS global_sum_2000
    FROM student_mobility
    WHERE year=2000
),
total_2024 AS (
    SELECT SUM(students) AS global_sum_2024
    FROM student_mobility
    WHERE year=2024
),
shares_2000 AS (
    SELECT 
        country,
        SUM(students) AS country_sum,
        SUM(students) * 1.0 / (SELECT global_sum_2000 
            FROM total_2000) AS share_2000
    FROM student_mobility
    WHERE year=2000
    GROUP BY country
),
shares_2024 AS (
    SELECT 
        country,
        SUM(students) AS country_sum,
        SUM(students) * 1.0 / (SELECT global_sum_2024 
            FROM total_2024) AS share_2024
    FROM student_mobility
    WHERE year=2024
    GROUP BY country
 )
SELECT 
    ROUND(SUM(share_2000 * share_2000), 4) AS HHI_2000,
    ROUND(SUM(share_2024 * share_2024), 4) AS HHI_2024,
    ROUND((SUM(share_2024 * share_2024)-SUM(share_2000 * share_2000))
        *100.0/NULLIF(SUM(share_2000 * share_2000),0),2) || '%' as Изменение
FROM shares_2000
    FULL OUTER JOIN shares_2024 USING(country);
```

Результат запроса сохранен в файл: <span style="color:red">12_hhi_2000_and_2024.csv</span>

In [13]:
hhi_2000_and_2024=pd.read_csv('12_hhi_2000_and_2024.csv')
hhi_2000_and_2024

,hhi_2000,hhi_2024,Изменение
0,0.0732,0.0983,34.15%


- **Доли Европы, Азии и Северной Америки за два периода (2000-2012 и 2013-2024 гг.)**
```
WITH reg AS(
    SELECT SUM(CASE WHEN region='Europe' AND year BETWEEN 2000 AND 2012 THEN students END) 
        AS sum_stud_e,
        SUM(CASE WHEN region='Europe' AND year BETWEEN 2013 AND 2024 THEN students END) 
        AS sum_stud_e_end,
        SUM(CASE WHEN region='Asia' AND year BETWEEN 2000 AND 2012 THEN students END) 
        AS sum_stud_a,
        SUM(CASE WHEN region='Asia' AND year BETWEEN 2013 AND 2024 THEN students END) 
        AS sum_stud_a_end,
        SUM(CASE WHEN region='North America' AND year BETWEEN 2000 AND 2012 
        THEN students END)AS sum_stud_nam,
        SUM(CASE WHEN region='North America' AND year BETWEEN 2013 AND 2024 THEN students END) 
        AS sum_stud_nam_end
    FROM student_mobility_regions
),
total AS(
    SELECT (
        SELECT SUM(students) 
        FROM student_mobility_regions
        WHERE year BETWEEN 2000 AND 2012
        ) AS total_2000_2012,
        (
        SELECT sum(students) 
        FROM student_mobility_regions
        WHERE year BETWEEN 2013 AND 2024
        ) AS total_2013_2024
)
SELECT '2000-2012' AS Период,
    ROUND(sum_stud_e*100.0/total_2000_2012,1) ||'%' AS "Доля стран Европы",
    ROUND(sum_stud_a*100.0/total_2000_2012,1) ||'%' AS "Доля стран Азии",
    ROUND(sum_stud_nam*100.0/total_2000_2012,1) ||'%' AS "Доля стран Северной Америки"
FROM reg CROSS JOIN total 
UNION ALL
SELECT '2013-2024' AS Период,
    ROUND(sum_stud_e_end*100.0/total_2013_2024,1) ||'%' AS "Доля стран Европы",
    ROUND(sum_stud_a_end*100.0/total_2013_2024,1) ||'%' AS "Доля стран Азии",
    ROUND(sum_stud_nam_end*100.0/total_2013_2024,1) ||'%' AS "Доля стран Северной Америки"
FROM reg CROSS JOIN total;
```

Результат запроса сохранен в файл: <span style="color:red">13_regions_share_2_periods.csv</span>

In [14]:
regions_share_2_periods=pd.read_csv('13_regions_share_2_periods.csv')
regions_share_2_periods

,Период,Доля стран Европы,Доля стран Азии,Доля стран Северной Америки
0,2000-2012,70.2%,22.8%,3.6%
1,2013-2024,71.7%,16.5%,9.4%


Структура мобильности в 2000 г.:  
Всего стран: 46  
Топ-5 стран: Франция, Великобритания, Турция, Казахстан и Финляндия (суммарная доля равна 52,7%)  
HHI: 0,0732  

Структура мобильности в 2024 г.:   
Всего стран: 55  
Топ-5 стран: Германия, Франция, Армения, Киргизия, Австрия (суммарная доля равна 52%)  
HHI: 0,0983  

Умеренный уровень концентрации мобильности (HHI) с распределением основной части потока между странами-лидерами (CR5 составляет 52-53%) и сохранением широкой географии направлений. Фиксируется тенденция к росту концентрации мобильности на протяжении рассматриваемого периода (HHI вырос на 34%).

Для оценки изменения региональной структуры мобильности были рассчитаны доли Европы, Азии и Северной Америки в общем потоке студентов за два периода: 2000–2012 и 2013–2024 гг., что соответствует расчёту среднегодовых темпов роста. Несмотря на то, что во втором периоде среднегодовой темп роста Азии (6,1%) значительно превышал европейский (–2,1%), доля Азии в общем потоке снизилась с 22,8% до 16,5%, а доля Европы выросла с 70,2% до 71,7%. Это говорит о высокой устойчивости региональной структуры мобильности: даже при смене трендов роста Европа сохраняет доминирующее положение, а диверсификация направлений происходит очень медленно.

## Ранжирование стран

- **Позиции стран по суммарному числу студентов в начале и конце рассматриваемого периода (2000 и 2024 гг.) и их изменение**
```
WITH rank_beg AS(
    SELECT country,
        RANK() OVER (ORDER BY SUM(students) DESC) AS rank_b
    FROM student_mobility
    WHERE year=(
        SELECT MIN(year)
        FROM student_mobility
    )
    GROUP BY country
),
rank_end AS(
    SELECT country,
        RANK() OVER (ORDER BY SUM(students) DESC) AS rank_e
    FROM student_mobility
    WHERE year=(
        SELECT MAX(year)
        FROM student_mobility
    )
    GROUP BY country
)
SELECT country AS Страна,
    rank_b AS Ранг_начало_периода,
    rank_e AS Ранг_конец_периода,
    CASE
        WHEN rank_b IS NULL OR rank_e IS NULL THEN '-'
        ELSE ABS(rank_e-rank_b) || ' п.'
    END AS Изменение_по_модулю,
    CASE
        WHEN rank_b>rank_e THEN 'Рост'
        WHEN rank_b<rank_e THEN 'Снижение'
        WHEN rank_b IS NULL OR rank_e IS NULL THEN 'Недостаточно данных'
        ELSE 'Без изменений'
    END AS Динамика_ранга	
FROM rank_beg FULL OUTER JOIN rank_end USING(country)
ORDER BY Ранг_конец_периода;
```

Результат запроса сохранен в файл: <span style="color:red">14_country_ranks_2000_and_2024.csv</span>

In [15]:
country_ranks_2000_and_2024=pd.read_csv('14_country_ranks_2000_and_2024.csv')
country_ranks_2000_and_2024.head(10)

,Страна,Ранг_начало_периода,Ранг_конец_периода,Изменение_по_модулю,Динамика_ранга
0,Germany,NaN,1.0,-,Недостаточно данных
1,France,1.0,2.0,1 п.,Снижение
2,Armenia,NaN,3.0,-,Недостаточно данных
3,Kyrgyzstan,NaN,4.0,-,Недостаточно данных
4,Austria,10.0,5.0,5 п.,Рост
5,Kazakhstan,4.0,6.0,2 п.,Снижение
6,Finland,5.0,7.0,2 п.,Снижение
7,Belarus,NaN,8.0,-,Недостаточно данных
8,"Korea, Republic of",25.0,9.0,16 п.,Рост
9,Slovakia,30.0,10.0,20 п.,Рост


*Рост:* Словакия (20 п.), Республика Корея (16 п.), Испания (10 п.), Австрия (5 п.), Литва (5 п.), Новая Зеландия (5 п.), Польша (1 п.).

*Снижение:* Норвегия (21 п.), Швеция (20 п.), Латвия (16 п.), Дания (14 п.), Куба (12 п.), Иордания (10 п.), Кипр (7 п.), Эстония (6 п.), Болгария (4 п.), Швейцария (3 п.), Монголия (3 п.), Чили (3 п.), Казахстан (2 п.), Австралия (2 п.), Финляндия (2 п.), Франция (1 п.), Малайзия (1 п.).

По остальным странам нет данных либо за 2000 г., либо за 2024 г.

## Итоги по странам

- **В итоговой таблице собраны основные показатели: число наблюдений, среднее число студентов, стандартное отклонение, коэффициент вариации, всего студентов, среднегодовой темп роста, ранг за период**
```
WITH years AS (
    SELECT 
        country,
        MIN(year) AS first_year,
        MAX(year) AS last_year
    FROM student_mobility
    GROUP BY country
),
stud AS(
    SELECT country,
        first_year,
        last_year,
        (SELECT students FROM student_mobility sm
        WHERE sm.country=y.country AND sm.year=y.first_year) AS first_count,
        (SELECT students FROM student_mobility sm
        WHERE sm.country=y.country AND sm.year=y.last_year) AS last_count
    FROM years y
),
cagr AS(
    SELECT country,
        first_year,
        last_year,
        CASE 
            WHEN first_count IS NULL OR first_count = 0 OR last_year = first_year
            THEN '—'
            ELSE ROUND((POWER(last_count*1.0/first_count, 1.0/
            (last_year - first_year))-1) * 100, 1) || '%' 
        END AS Среднегодовой_темп_роста
    FROM stud
),
stats AS(
    SELECT country,
        COUNT(*) AS Число_наблюдений,
        ROUND(AVG(students)) AS Среднее_число_студентов,
        ROUND(STDDEV(students),1) AS Станд_отклонение,
        ROUND(STDDEV(students)/AVG(students),2) AS Коэффициент_вариации,
        SUM(students) AS Всего_студентов,
        RANK() OVER (ORDER BY SUM(students) DESC) AS Ранг_за_период
    FROM student_mobility
    GROUP BY country
)
SELECT country as Страна,
    Число_наблюдений,
    Среднее_число_студентов,
    Станд_отклонение,
    Коэффициент_вариации,
    Всего_студентов,
    Среднегодовой_темп_роста,
    Ранг_за_период
FROM stats JOIN cagr USING (country)
ORDER BY Ранг_за_период, Страна;
```

Результат запроса сохранен в файл: <span style="color:red">15_countries_summary.csv</span>

In [16]:
countries_summary=pd.read_csv('15_countries_summary.csv')
countries_summary.head(10)

,Страна,Число_наблюдений,Среднее_число_студентов,Станд_отклонение,Коэффициент_вариации,Всего_студентов,Среднегодовой_темп_роста,Ранг_за_период
0,Germany,12,10314,730.6,0.07,123770,1.3%,1
1,Czechia,24,3329,2742.1,0.82,79907,20.3%,2
2,France,25,3064,716.6,0.23,76612,3.3%,3
3,United Kingdom,24,2909,926.6,0.32,69804,4.7%,4
4,United States,10,4930,285.2,0.06,49296,-0.9%,5
5,Ukraine,19,2484,1566.7,0.63,47199,-8.7%,6
6,Finland,24,1763,648.4,0.37,42314,4.9%,7
7,Kazakhstan,21,1858,811.3,0.44,39012,4.6%,8
8,Belarus,23,1533,980.2,0.64,35264,—,9
9,Italy,24,1400,845.3,0.60,33592,15.2%,10


## Основные результаты

- На протяжении 25 лет Европа остается ведущим направлением международной мобильности российских студентов.
- Абсолютный лидер по входящей мобильности студентов из России - Германия.
- Наиболее стабильные направления - США и Германия. Наименее стабильные направления среди ведущих стран Европы и Азии – Чехия, Киргизия, Турция и Республика Корея.
- Чехия, Италия, Армения и Республика Корея показывают наиболее высокие среднегодовые темпы роста за весь период с 2000 по 2024 г., но актуальные лидеры роста в Европе и Азии (2013-2024 гг.) – Турция, Словакия, Сербия, Словения, Венгрия, Узбекистан, Республика Корея.
- Концентрация мобильности по странам (HHI) умеренная с распределением основной части потока между странами-лидерами и сохранением широкой географии направлений (110 стран). Фиксируется тенденция к росту концентрации мобильности (HHI за 25 лет вырос на 34%).
- Наблюдается изменение тренда мобильности: во второй половине рассматриваемого периода Европа от быстрого роста переходит к снижению, а Азия продолжает расти, но темп роста снижается. В 2000-2024 гг. среднегодовой темп роста стран Азии (7%) выше, чем стран Европы (6,4%).
- Региональная структура международной мобильности российских студентов очень устойчива: несмотря на смену тренда, доля стран Европы в общем потоке российских студентов за границу остается значительно больше доли стран Азии. Это связано с более высоким исходным уровнем и масштабом образовательных потоков в Европу.